# Gemini Crypto Data Downloader

Downloads public Gemini candle data and stores it by symbol under `./data/<symbol>/`.

In [7]:
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import requests


In [8]:
# ---- Config ----
BASE_URL = "https://api.gemini.com"
SYMBOLS = ["btcusd", "ethusd", "solusd"]  # Gemini symbols are lowercase
TIMEFRAMES = ["1m", "5m", "1h", "1day"]
OUTPUT_ROOT = Path("./data")
SAVE_PARQUET = True
REQUEST_TIMEOUT = 20


In [ ]:
import time
import csv
from pathlib import Path
from typing import Optional

import requests

BASE_URL = "https://api.gemini.com"
# BASE_URL = "https://api.sandbox.gemini.com"

def gemini_get_json(session: requests.Session, path: str, params=None, max_retries: int = 8):
    """GET with basic retry/backoff (handles 429/5xx)."""
    url = BASE_URL + path
    backoff = 0.5
    last_exc = None

    for _ in range(max_retries):
        try:
            r = session.get(url, params=params or {}, timeout=30)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(backoff)
                backoff = min(backoff * 2, 10.0)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_exc = e
            time.sleep(backoff)
            backoff = min(backoff * 2, 10.0)

    raise RuntimeError(f"Failed GET {url}") from last_exc

def list_symbols(session: requests.Session):
    return gemini_get_json(session, "/v1/symbols")

def fetch_candles(session: requests.Session, symbol: str, timeframe: str):
    return gemini_get_json(session, f"/v2/candles/{symbol}/{timeframe}")

def normalize_filter_sort(candles, cutoff_ms: int):
    """
    Each candle: [timestamp_ms, open, high, low, close, volume]
    Filter to last N days and sort ascending by timestamp.
    """
    out = []
    for row in candles:
        if not isinstance(row, (list, tuple)) or len(row) < 6:
            continue
        ts = int(row[0])
        if ts >= cutoff_ms:
            out.append([ts, float(row[1]), float(row[2]), float(row[3]), float(row[4]), float(row[5])])
    out.sort(key=lambda x: x[0])
    return out

def write_data_file(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["timestamp_ms", "open", "high", "low", "close", "volume"])
        w.writerows(rows)

def download_all_symbols_ohlcv(
    timeframe: str = "1m",
    days: int = 7,
    out_dir: str = "./data",
    max_symbols: Optional[int] = None,
    sleep_between: float = 0.05,
):
    cutoff_ms = int((time.time() - days * 24 * 3600) * 1000)
    out_path = Path(out_dir) / "gemini" / f"ohlcv_{timeframe}_{days}d"
    out_path.mkdir(parents=True, exist_ok=True)

    with requests.Session() as session:
        symbols = list_symbols(session)
        if max_symbols is not None:
            symbols = symbols[:max_symbols]

        (out_path / "symbols.txt").write_text("\n".join(symbols))

        ok, failed = 0, 0
        for i, sym in enumerate(symbols, 1):
            try:
                candles = fetch_candles(session, sym, timeframe)
                rows = normalize_filter_sort(candles, cutoff_ms)

                # one file per symbol with ".data" extension (CSV content)
                file_path = out_path / f"{sym}.data"
                write_data_file(file_path, rows)

                ok += 1
                if i % 25 == 0 or i == len(symbols):
                    print(f"[{i}/{len(symbols)}] ok={ok} failed={failed} latest={sym} rows={len(rows)}")

            except Exception as e:
                failed += 1
                print(f"[{i}/{len(symbols)}] FAILED {sym}: {type(e).__name__}: {e}")

            time.sleep(sleep_between)

    print(f"Done. Files at: {out_path.resolve()}")

# ---- run ----
if __name__ == "__main__":
    download_all_symbols_ohlcv(
        timeframe="1m",   # highest frequency candles
        days=7,
        out_dir="./.data",
        max_symbols=None,  # set e.g. 50 to test quickly
        sleep_between=0.05
    )

[25/451] ok=25 failed=0 latest=api3gusd rows=1440
[50/451] ok=50 failed=0 latest=batusdc rows=1440
[75/451] ok=75 failed=0 latest=btcgusd rows=1440
[100/451] ok=100 failed=0 latest=ctxrlusd rows=1440
[125/451] ok=125 failed=0 latest=driftusd rows=1440
[127/451] FAILED efilfil: RuntimeError: Failed GET https://api.gemini.com/v2/candles/efilfil/1m
[150/451] ok=149 failed=1 latest=eulusdc rows=1440
[175/451] ok=174 failed=1 latest=galagusd rows=1440
[193/451] FAILED gusdusd: RuntimeError: Failed GET https://api.gemini.com/v2/candles/gusdusd/1m
[200/451] ok=198 failed=2 latest=hyperlusd rows=1440
[225/451] ok=223 failed=2 latest=jtousd rows=1440
[250/451] ok=248 failed=2 latest=linkusdc rows=1440
[275/451] ok=273 failed=2 latest=maskusdc rows=1439
[300/451] ok=298 failed=2 latest=pengugusd rows=1440
[325/451] ok=323 failed=2 latest=popcatusdcperp rows=1440
[350/451] ok=348 failed=2 latest=sandrlusd rows=1440
[375/451] ok=373 failed=2 latest=storjgusd rows=1440
[400/451] ok=398 failed=2 lat